# Lesson 3, one call at a time

Lesson 3 fitted a straight line to house prices three ways: by the exact formula
(the normal equation), by gradient descent, and with a penalty on large coefficients
(Ridge and Lasso). This notebook takes each step on two to four houses, small enough to
do with a pencil, and recomputes the worked examples of lesson 3's commentary
(`Docs/regression_commentary.md`) with NumPy and scikit-learn, so that the arithmetic
by hand and the library call sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next cell*:
answer it first, then run. Notebook `00_the_scikit_learn_api.ipynb` explains `fit`,
`predict`, `score` and the shape of `X`; this one assumes it.

The notation is lesson 3's: $x$ an input, $y$ the true price, $\hat y$ the prediction,
$w$ a coefficient, $b$ the intercept, $\theta$ all parameters together, $m$ the number
of houses, $J$ the cost, $\alpha$ the learning rate, $\lambda$ the penalty strength
(which scikit-learn calls `alpha`).

Not examinable. The full-scale version of each atom is in lesson 3's notebooks
`01_linear_regression_from_scratch.ipynb`, `02_polynomial_and_overfitting.ipynb` and
`03_ridge_and_lasso.ipynb`.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1

import sys

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

np.set_printoptions(precision=4, suppress=True)

## 1. The best single price for three houses

Before any feature, the simplest model predicts one number for every house. Which
number is best depends on how errors are charged: the mean squared error (MSE) or the
mean absolute error (MAE). `DummyRegressor` is that model, the regression baseline.

*From:* lesson 3, slide *Why squared, not absolute*; commentary, worked example *the best single price for three houses*.

In [2]:
prices = np.array([150.0, 200.0, 700.0])       # thousands of euros; the third is a mansion
X_none = np.zeros((3, 1))                         # a dummy ignores the inputs, but fit still wants an X

### Your prediction, before you run the next cell

Which single guess minimises the squared error, and which the absolute
error: 200 (the middle house) or 350 (the average)?

In [3]:
for strategy in ("mean", "median"):
    guess = DummyRegressor(strategy=strategy).fit(X_none, prices).predict(X_none)
    print(f"{strategy:6}: guess {guess[0]:.0f}   MSE {mean_squared_error(prices, guess):,.0f}"
          f"   MAE {mean_absolute_error(prices, guess):.1f}")

grid = np.arange(100, 801)
mse = [((prices - c) ** 2).mean() for c in grid]
mae = [np.abs(prices - c).mean() for c in grid]
print("searching every guess from 100 to 800: MSE smallest at", grid[np.argmin(mse)],
      "  MAE smallest at", grid[np.argmin(mae)])

mean  : guess 350   MSE 61,667   MAE 233.3
median: guess 200   MSE 84,167   MAE 183.3
searching every guess from 100 to 800: MSE smallest at 350   MAE smallest at 200


**What to look at.** Squared error is smallest at the **mean**, 350 (MSE 61,667);
absolute error at the **median**, 200 (MAE 183.3). The mansion drags the squared-error
answer 150 thousand above the middle house, past two of the three. Squaring charges a
big error far more than a small one, so the worst house gets the most say.

**Try changing.** make the mansion 1,500 and see which of the two answers moves.

## 2. The cost of one guess

The cost of lesson 3 is $J = \frac{1}{2m}\sum_i (\hat y^{(i)} - y^{(i)})^2$, half
the mean squared error. The ½ only makes the derivative tidier; it does not move the
minimum. One guess of $w$ and $b$ gives one value of $J$.

*From:* lesson 3, slides *The cost function: mean squared error* and *Trying ŷ = 2,400 · area + 45,000*.

In [4]:
area = np.array([80.0, 120.0, 200.0])                  # square metres
price = np.array([240_000.0, 320_000.0, 540_000.0])     # euros
m = len(area)

### Your prediction, before you run the next cell

With $w$ = 2,400 euros per square metre and $b$ = 45,000 euros, which house
has the largest error, and roughly what is $J$?

In [5]:
def cost(w, b):
    errors = (w * area + b) - price
    return (errors ** 2).sum() / (2 * m), errors

J, errors = cost(2_400, 45_000)
print("errors ŷ - y:", errors)
print("squared:     ", errors ** 2)
print(f"J = {J:.3e}")
J_fit, _ = cost(2_535.714, 28_571.43)        # the least-squares line of atom 5
print(f"J at the fitted line = {J_fit:.3e}")

errors ŷ - y: [ -3000.  13000. -15000.]
squared:      [9.00e+06 1.69e+08 2.25e+08]
J = 6.717e+07
J at the fitted line = 4.286e+07


**What to look at.** Errors of −3,000, +13,000 and −15,000 euros; squared, the two
large ones are 1.69 × 10⁸ and 2.25 × 10⁸, so the house at 80 m² barely counts. $J$ is
about **6.7 × 10⁷**, a number with no meaning on its own: it means something only next
to another cost, like the line of atom 5, which is lower. Fitting is the search for
the $w$ and $b$ with the lowest $J$.

**Try changing.** try $w$ = 2,500 with $b$ = 45,000 and see whether $J$ goes up or
down.

## 3. R², by hand and with `r2_score`

The coefficient of determination $R^2 = 1 - \text{SS}_\text{res} / \text{SS}_\text{tot}$
compares the model's squared errors ($\text{SS}_\text{res}$) with those of predicting
the mean for every house ($\text{SS}_\text{tot}$). 1 means no error; 0 means no better
than the mean.

*From:* lesson 3, slide *A cost needs something to be compared against*.

In [6]:
y_true = np.array([200.0, 250.0, 300.0, 450.0])     # four test houses, thousands of euros
y_pred = np.array([210.0, 240.0, 320.0, 430.0])     # the model's predictions

### Your prediction, before you run the next cell

The errors are 10, 10, 20 and 20 thousand. Is $R^2$ closer to 0.5 or to
1?

In [7]:
ss_res = ((y_true - y_pred) ** 2).sum()
ss_tot = ((y_true - y_true.mean()) ** 2).sum()
print(f"SS_res = {ss_res:.0f}   SS_tot = {ss_tot:.0f}   R² = 1 - {ss_res:.0f}/{ss_tot:.0f} = {1 - ss_res / ss_tot:.3f}")
print("r2_score:", round(r2_score(y_true, y_pred), 3))

SS_res = 1000   SS_tot = 35000   R² = 1 - 1000/35000 = 0.971
r2_score: 0.971


**What to look at.** $1 - 1000/35000$ = **0.971**, by hand and from `r2_score`, which
takes the true values first. In scikit-learn, `model.score(X, y)` on a regressor returns
this same number.

**Try changing.** predict the mean, 300, for all four houses and check that $R^2$ is
exactly 0.

## 4. The same model, two test sets

$R^2$ is a ratio to the test set's own spread, so it depends on the test set as
much as on the model. The root mean squared error (RMSE), in euros, does not.

*From:* lesson 3, slide *A cost needs something to be compared against*; commentary, worked example *the same model, two test sets*.

In [8]:
varied = (np.array([200.0, 250.0, 300.0, 450.0]), np.array([210.0, 240.0, 320.0, 430.0]))
lookalike = (np.array([290.0, 300.0, 310.0, 300.0]), np.array([295.0, 305.0, 305.0, 300.0]))

### Your prediction, before you run the next cell

On the look-alike houses the model's errors are smaller. Will $R^2$ be
higher or lower than on the varied houses?

In [9]:
for name, (yt, yp) in (("varied", varied), ("look-alike", lookalike)):
    rmse = np.sqrt(mean_squared_error(yt, yp))
    rmse_mean = np.sqrt(mean_squared_error(yt, np.full_like(yt, yt.mean())))
    print(f"{name:10}: RMSE model {rmse:5.1f}   RMSE predicting the mean {rmse_mean:5.1f}   R² {r2_score(yt, yp):.3f}")

varied    : RMSE model  15.8   RMSE predicting the mean  93.5   R² 0.971
look-alike: RMSE model   4.3   RMSE predicting the mean   7.1   R² 0.625


**What to look at.** The model's error fell from **15.8** to **4.3** thousand euros,
and $R^2$ fell too, from **0.971** to **0.625**: the look-alike houses leave the
mean-predicting baseline almost nothing to get wrong (7.1). Report the RMSE, in the
units of the problem, beside $R^2$.

**Try changing.** add a fifth look-alike house at 400 with prediction 395.

## 5. The design matrix, the label vector, and the normal equation by hand

The **design matrix** $X$ has one row per house and one column per feature, plus a
column of 1s whose coefficient is the intercept $b$. The **label vector** $y$ holds the
prices. With $\theta = (b, w)$, every prediction at once is $X\theta$, and the
least-squares $\theta$ solves the normal equation $X^\top X\,\theta = X^\top y$.

*From:* lesson 3, slides *θ and X: all parameters, all houses*, *The normal equation* and *The same equation, by hand*.

In [10]:
X = np.column_stack([np.ones(3), area])      # column of 1s for b, then the areas
y = price
print("X =\n", X)
print("y =", y)

X =
 [[  1.  80.]
 [  1. 120.]
 [  1. 200.]]
y = [240000. 320000. 540000.]


### Your prediction, before you run the next cell

The slope that generated these prices was 2,400 euros per square metre.
Will three houses give a slope within 10% of it?

In [11]:
XtX = X.T @ X
Xty = X.T @ y
print("XᵀX =\n", XtX)
print("Xᵀy =", Xty)
print("determinant of XᵀX:", round(np.linalg.det(XtX)))
theta = np.linalg.solve(XtX, Xty)            # solve the system; never invert by hand in code
print(f"θ = (b, w) = ({theta[0]:,.0f}, {theta[1]:,.1f})")

lr = LinearRegression().fit(area.reshape(-1, 1), price)
print(f"LinearRegression: intercept_ {lr.intercept_:,.0f}, coef_ {lr.coef_[0]:,.1f}")

XᵀX =
 [[    3.   400.]
 [  400. 60800.]]
Xᵀy = [1.100e+06 1.656e+08]
determinant of XᵀX: 22400
θ = (b, w) = (28,571, 2,535.7)
LinearRegression: intercept_ 28,571, coef_ 2,535.7


**What to look at.** $X^\top X$ has rows (3, 400) and (400, 60,800), $X^\top y$ is
(1,100,000, 165,600,000), the determinant is 22,400, and $\theta$ gives a slope of
**2,536** euros per square metre and an intercept of **28,571** euros: the slide's
numbers (the slide rounds the intercept to 28,600 and writes prices in thousands, so
its $X^\top y$ is 1,000 times smaller). `LinearRegression` gives the same two, stored in `intercept_` and `coef_`. The
6% distance from 2,400 is luck on three houses, not accuracy.

**Try changing.** add a fourth house, 150 m² at 380,000 euros, and see how far the
slope moves.

## 6. One slope, no intercept: the cost is a bowl

With one coefficient and no intercept, $\hat y = wx$, the cost $J(w)$ is a parabola,
and setting its derivative to zero gives $w^* = \sum xy / \sum x^2$.

*From:* lesson 3, slides *The cost is a bowl* and *And it really is the minimum*; commentary, worked example *one slope, no intercept, three houses*.

In [12]:
x = np.array([1.0, 2.0, 3.0])      # areas, hundreds of square metres
y3 = np.array([2.0, 3.0, 6.0])     # prices, hundreds of thousands of euros

### Your prediction, before you run the next cell

What is $w^* = \sum xy / \sum x^2$? And if $w$ moves 0.3 above or 0.3 below
it, does the cost rise by the same amount on both sides?

In [13]:
w_star = (x * y3).sum() / (x * x).sum()
print(f"w* = {(x * y3).sum():.0f} / {(x * x).sum():.0f} = {w_star:.3f}")
print("LinearRegression(fit_intercept=False):", LinearRegression(fit_intercept=False).fit(x.reshape(-1, 1), y3).coef_.round(3))

J = lambda w: ((w * x - y3) ** 2).sum() / (2 * len(x))
for w in (w_star - 0.6, w_star - 0.3, w_star, w_star + 0.3, w_star + 0.6):
    print(f"w = {w:.3f}   J = {J(w):.4f}")
print("rise for a step of 0.3, either side:", round(J(w_star + 0.3) - J(w_star), 4), round(J(w_star - 0.3) - J(w_star), 4))
print("curvature Σx²/m:", round((x * x).sum() / len(x), 3))

w* = 26 / 14 = 1.857
LinearRegression(fit_intercept=False): [1.857]
w = 1.257   J = 0.9590
w = 1.557   J = 0.3290
w = 1.857   J = 0.1190
w = 2.157   J = 0.3290
w = 2.457   J = 0.9590
rise for a step of 0.3, either side: 0.21 0.21
curvature Σx²/m: 4.667


**What to look at.** $w^* = 26/14 = $ **1.857**, and `fit_intercept=False` gives the
same. A step of 0.3 either side raises the cost by exactly **0.21** both ways: the cost
is a parabola with curvature $\sum x^2/m = 4.667$, which can never be negative, so the
flat point is the bottom of a bowl, not the top of a hill.

**Try changing.** add an intercept back with `LinearRegression()` and compare the
slope.

## 7. Two standardised features, rising correlation

With two standardised features correlated $r$, the curvatures of the cost in its
steepest and shallowest directions are $1 + r$ and $1 - r$, the eigenvalues of the
matrix with 1 on the diagonal and $r$ off it. Their ratio $\kappa = (1+r)/(1-r)$, the
**condition number**, says how stretched the valley is, and gradient descent needs
roughly $\kappa$ times as many steps as on a round bowl.

*From:* lesson 3, slide *How stretched is the valley?*; commentary, worked example *two standardised features, rising correlation*.

### Your prediction, before you run the next cell

At $r = 0.99$, how stretched is the valley: 2, 20, or 200?

In [14]:
for r in (0.0, 0.5, 0.9, 0.99):
    eig = np.linalg.eigvalsh(np.array([[1, r], [r, 1]]))
    print(f"r = {r:4}:  curvatures {eig.max():.2f} and {eig.min():.2f}   κ = {eig.max() / eig.min():.0f}")

r =  0.0:  curvatures 1.00 and 1.00   κ = 1
r =  0.5:  curvatures 1.50 and 0.50   κ = 3
r =  0.9:  curvatures 1.90 and 0.10   κ = 19
r = 0.99:  curvatures 1.99 and 0.01   κ = 199


**What to look at.** κ is **1, 3, 19 and 199**. Standardising fixes the units, not
the correlation: two features that nearly say the same thing still make a stretched
valley.

### Your prediction, before you run the next cell

Is that only a formula? Draw 10,000 houses with two features correlated
0.9, standardise them, and compute the curvatures of the cost from the data itself.

In [15]:
rng = np.random.default_rng(0)
f1 = rng.normal(size=10_000)
f2 = 0.9 * f1 + np.sqrt(1 - 0.9 ** 2) * rng.normal(size=10_000)
Z = StandardScaler().fit_transform(np.column_stack([f1, f2]))
eig = np.linalg.eigvalsh(Z.T @ Z / len(Z))      # the curvatures of the cost J = (1/2m)·Σ errors²
print(f"measured correlation {np.corrcoef(f1, f2)[0, 1]:.3f}   curvatures {eig.max():.3f}, {eig.min():.3f}   κ {eig.max() / eig.min():.1f}")

measured correlation 0.900   curvatures 1.900, 0.100   κ 18.9


**What to look at.** From the data, the curvatures come out close to 1.9 and 0.1
and κ close to 19: the formula and the sample agree, to the noise of a sample.

**Try changing.** set the correlation to 0.99 in the second cell.

## 8. Three houses voting on the price of a square metre

The gradient of the cost for the area coefficient is the average of
(error × area): $\partial J/\partial w = \frac1m \sum_i (\hat y^{(i)} - y^{(i)})\,x^{(i)}$. Each house
pulls the coefficient in proportion to how wrong it was *and* how large it is.

*From:* lesson 3, slide *The gradients: one for w, one for b*; commentary, worked example *three houses voting on the price of a square metre*.

In [16]:
area3 = np.array([80, 120, 200])     # square metres
err3 = np.array([10, -5, -20])         # prediction minus truth, thousands of euros

### Your prediction, before you run the next cell

Will the gradient be positive or negative, and which house supplies most
of it?

In [17]:
pulls = err3 * area3
print("pulls, error × area:", pulls)
grad = pulls.mean()
print(f"∂J/∂w = ({pulls[0]:+} {pulls[1]:+} {pulls[2]:+}) / 3 = {grad:+.1f}")
print(f"share of the total pull from the 200 m² house: {abs(pulls[2]) / np.abs(pulls).sum():.0%}")

pulls, error × area: [  800  -600 -4000]
∂J/∂w = (+800 -600 -4000) / 3 = -1266.7
share of the total pull from the 200 m² house: 74%


**What to look at.** The gradient is **−1,266.7**: negative, so the update
$w \leftarrow w - \alpha\,\partial J/\partial w$ **raises** $w$, because the model was
charging too little per square metre. The big house supplies **74%** of the total
pull: it was the most wrong *and* the largest.

**Try changing.** give the 80 m² house the error −20 instead and compare its pull
with the big house's.

## 9. Gradient descent, one iteration at a time

One iteration: predict with the current $w$ and $b$, measure every error, average
them into the two gradients, step both parameters against their gradients by the
learning rate $\alpha$, and start again with the new line.

*From:* lesson 3, slides *One iteration, then the next*, *One step on two houses, from w = 0 and b = 0* and *The path it takes*.

In [18]:
x2 = np.array([80.0, 200.0])       # square metres
y2 = np.array([240.0, 540.0])      # thousands of euros
alpha = 1e-5

### Your prediction, before you run the next cell

Starting from $w = 0$, $b = 0$: what are the two gradients, and after one
step, which parameter has moved further?

In [19]:
w, b = 0.0, 0.0
for it in range(1, 4):
    y_hat = w * x2 + b                       # 1. predict
    error = y_hat - y2                       # 2. measure every error
    grad_w = (error * x2).mean()             # 3. average into the two gradients
    grad_b = error.mean()
    w, b = w - alpha * grad_w, b - alpha * grad_b    # 4. step both, with the old gradients
    print(f"iteration {it}: grad_w {grad_w:10,.1f}  grad_b {grad_b:8,.1f}  ->  w {w:.4f}  b {b:.6f}")

iteration 1: grad_w  -63,600.0  grad_b   -390.0  ->  w 0.6360  b 0.003900
iteration 2: grad_w  -48,844.3  grad_b   -301.0  ->  w 1.1244  b 0.006910
iteration 3: grad_w  -37,512.0  grad_b   -232.6  ->  w 1.4996  b 0.009235


**What to look at.** First iteration: gradients **−63,600** and **−390**, so after one
step $w$ = **0.636** and $b$ = **0.0039**, the slide's numbers. The slope moved 163 times
further than the intercept, because its gradient multiplies each error by the area (80
and 200) where the intercept's multiplies it by 1. Each later iteration starts from the
new line, so every error and both gradients are recomputed.

### Your prediction, before you run the next cell

The exact line through the two houses has $w = 2.5$ and $b = 40$. After
4,000 iterations, how close will gradient descent be to each?

In [20]:
w, b = 0.0, 0.0
for it in range(4_000):
    error = (w * x2 + b) - y2
    w, b = w - alpha * (error * x2).mean(), b - alpha * error.mean()
print(f"after 4,000 iterations: w = {w:.3f}  b = {b:.3f}")
print("exact:", LinearRegression().fit(x2.reshape(-1, 1), y2).coef_, LinearRegression().fit(x2.reshape(-1, 1), y2).intercept_)

after 4,000 iterations: w = 2.740  b = 0.264
exact: [2.5] 40.0


**What to look at.** The intercept has barely left zero, 0.264 of its 40, and the
slope has settled at 2.740 instead of 2.5: that is the best slope for a line forced
through zero, so the slope is making up for the intercept that has not arrived. With
one shared learning rate and unscaled areas, the intercept crawls. Standardising the area, as lesson 2 did, is the cure, and lesson 3's notebook 1
shows it on 450 houses.

**Try changing.** standardise `x2` first, `(x2 - x2.mean()) / x2.std()`, use
`alpha = 0.5`, and see how few iterations are needed.

## 10. A bowl of curvature 2, and four learning rates

On $J(w) = \tfrac12 \cdot 2\,w^2$ the gradient is $2w$, so each step multiplies $w$ by
$1 - 2\alpha$. Whether the path creeps, overshoots or explodes depends only on that
factor.

*From:* lesson 3, slide *Three learning rates*; commentary, worked example *a bowl of curvature 2*.

### Your prediction, before you run the next cell

Starting at $w = 1$, which of the four learning rates 0.1, 0.5, 0.9 and
1.1 will move away from the minimum at 0?

In [21]:
for alpha_try in (0.1, 0.5, 0.9, 1.1):
    w = 1.0
    path = []
    for _ in range(3):
        w = w - alpha_try * (2 * w)
        path.append(round(w, 3))
    print(f"α = {alpha_try}:  factor {1 - 2 * alpha_try:+.1f}   first three steps {path}")

α = 0.1:  factor +0.8   first three steps [0.8, 0.64, 0.512]
α = 0.5:  factor +0.0   first three steps [0.0, 0.0, 0.0]
α = 0.9:  factor -0.8   first three steps [-0.8, 0.64, -0.512]
α = 1.1:  factor -1.2   first three steps [-1.2, 1.44, -1.728]


**What to look at.** 0.1 creeps in from one side (×0.8); 0.5 lands on the minimum in
one step (×0); 0.9 overshoots and oscillates but closes in (×−0.8); **1.1** overshoots
further every step (×−1.2) and diverges. The limit is $2/c = 1$ for curvature $c = 2$;
with several features, the steepest direction sets $c$ for all of them.

**Try changing.** find the learning rate at which the path oscillates forever without
moving away or closing in.

## 11. A parabola by the normal equation

A linear model is linear in its coefficients, not in its inputs. Add a column $t^2$
to the design matrix and the same normal equation fits a curve.

*From:* lesson 3, slide *"Linear" means linear in the coefficients*; commentary, worked example *a parabola by the normal equation*.

In [22]:
t = np.array([-1.0, 0.0, 1.0])      # temperature, standardised
energy = np.array([3.0, 1.0, 3.0])

### Your prediction, before you run the next cell

A straight line through these three points is flat at 7/3. What $\theta$ will
the normal equation give with a column of $t^2$?

In [23]:
Xp = np.column_stack([np.ones(3), t, t ** 2])
theta = np.linalg.solve(Xp.T @ Xp, Xp.T @ energy)
print("X =\n", Xp)
print("θ =", theta)

poly = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression()).fit(t.reshape(-1, 1), energy)
print("PolynomialFeatures(2) + LinearRegression: intercept_", poly[-1].intercept_.round(3), " coef_", poly[-1].coef_.round(3))

X =
 [[ 1. -1.  1.]
 [ 1.  0.  0.]
 [ 1.  1.  1.]]
θ = [1. 0. 2.]
PolynomialFeatures(2) + LinearRegression: intercept_ 1.0  coef_ [0. 2.]


**What to look at.** $\theta = (1, 0, 2)$, so $\hat y = 1 + 2t^2$: the U through all
three points. `PolynomialFeatures(degree=2)` builds the columns $t$ and $t^2$ (with
`include_bias=False` it leaves the column of 1s to `LinearRegression`), and the result
is the same. Nothing in the method changed: same equation, one more column.

**Try changing.** use `degree=5` on these three points: more coefficients than
houses, and the fit is no longer unique.

## 12. What a penalty buys

A degree-12 polynomial on 21 days of energy data can follow every point, with
coefficients in the billions. Ridge regression adds $\lambda \sum_j w_j^2$ to the cost, so
large coefficients cost something. This cell is lesson 3 notebook 3's own code, with the
same seed, so it reproduces the slide's table.

*From:* lesson 3, slides *Charge for size* and *What a penalty buys*; notebook 3, section *Ridge against the degree-12 disaster*.

In [24]:
rng = np.random.default_rng(42)
n = 30
temperature = rng.uniform(-5, 35, n)
consumption = 240 + 1.15 * (temperature - 18) ** 2 + rng.normal(0, 22, n)    # noise of 22 kWh
t_train, t_test, c_train, c_test = train_test_split(temperature, consumption, test_size=0.3, random_state=42)
print(len(t_train), "training days,", len(t_test), "test days")

21 training days, 9 test days


### Your prediction, before you run the next cell

Without a penalty the largest coefficient is about three billion. What
will a penalty of $\lambda = 0.01$ leave, and what happens to the training error?

In [25]:
rows = []
for label, estimator in [("none", LinearRegression()), ("λ = 0.01", Ridge(alpha=0.01)),
                         ("λ = 1", Ridge(alpha=1)), ("λ = 100", Ridge(alpha=100))]:
    model = make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), estimator)
    model.fit(t_train.reshape(-1, 1), c_train)
    rows.append({"penalty": label,
                 "train RMSE": np.sqrt(mean_squared_error(c_train, model.predict(t_train.reshape(-1, 1)))),
                 "test RMSE": np.sqrt(mean_squared_error(c_test, model.predict(t_test.reshape(-1, 1)))),
                 "largest |w|": np.abs(model[-1].coef_).max()})
table = pd.DataFrame(rows).set_index("penalty").round(1)
table["largest |w|"] = table["largest |w|"].map("{:,.1f}".format)
table

,train RMSE,test RMSE,largest |w|
penalty,,,
none,13.6,24655.7,"3,097,038,010.0"
λ = 0.01,18.0,22.8,364.7
λ = 1,44.5,105.2,156.5
λ = 100,96.0,227.6,6.3


**What to look at.** The largest coefficient falls from 3,097,038,010 to **365**,
and the test RMSE from 24,655.7 to 22.8 kilowatt-hours, the size of the noise (22). The
training error rises at every step, 13.6, 18.0, 44.5, 96.0: a worse fit on the data we
have, traded for a better one on new data. Too much penalty loses again, 227.6 at
$\lambda = 100$, which is why $\lambda$ is searched for, not guessed.

**Try changing.** try `Ridge(alpha=0.1)` and `Ridge(alpha=10)`.

## 13. One coefficient, four penalties

Ridge's exact solution is $\theta = (X^\top X + \lambda I)^{-1} X^\top y$. With one
feature and no intercept it is a division: $w = \sum xy / (\sum x^2 + \lambda)$.

*From:* lesson 3, slide *Ridge has an exact solution too*; commentary, worked example *one coefficient, four penalties*.

### Your prediction, before you run the next cell

On the houses of atom 6 ($\sum xy = 26$, $\sum x^2 = 14$), what is $w$ at
$\lambda = 14$? And at $\lambda = 100$, is it zero?

In [26]:
for lam in (0, 1, 14, 100):
    by_formula = 26 / (14 + lam)
    by_sklearn = Ridge(alpha=lam, fit_intercept=False).fit(x.reshape(-1, 1), y3).coef_[0]
    print(f"λ = {lam:3}:  26/(14 + {lam}) = {by_formula:.3f}   Ridge: {by_sklearn:.3f}")

λ =   0:  26/(14 + 0) = 1.857   Ridge: 1.857
λ =   1:  26/(14 + 1) = 1.733   Ridge: 1.733
λ =  14:  26/(14 + 14) = 0.929   Ridge: 0.929
λ = 100:  26/(14 + 100) = 0.228   Ridge: 0.228


**What to look at.** 1.857, 1.733, **0.929**, 0.228. $\lambda = \sum x^2 = 14$ halves the
coefficient; no finite $\lambda$ makes it zero. scikit-learn's `Ridge(alpha=λ)`
minimises $\sum (\hat y - y)^2 + \lambda \sum w^2$, exactly this formula, so the two
columns agree.

**Try changing.** compare with `Lasso(alpha=1, fit_intercept=False)` on the same data,
which can reach zero.

## 14. The same column twice: one effect, two columns

Put the same column in twice and $X^\top X$ has no inverse: least squares fixes only
the sum $w_1 + w_2$, and every split of it predicts the same prices. Adding $\lambda$ down
the diagonal restores the inverse, and Ridge picks the even split.

*From:* lesson 3, slides *Why that one change fixes it* and *The case Ridge was invented for*; commentary, worked examples *the same column twice* and *one effect, two columns*.

In [27]:
X_dup = np.column_stack([x, x])       # the area of atom 6, twice
print("XᵀX =\n", X_dup.T @ X_dup)

XᵀX =
 [[14. 14.]
 [14. 14.]]


### Your prediction, before you run the next cell

What is the determinant of $X^\top X$, and after adding $\lambda = 1$ to its
diagonal?

In [28]:
A = X_dup.T @ X_dup
print("det(XᵀX)      =", round(np.linalg.det(A), 6), "  eigenvalues", np.linalg.eigvalsh(A).round(6))
print("det(XᵀX + 1·I) =", round(np.linalg.det(A + np.eye(2)), 6), "  eigenvalues", np.linalg.eigvalsh(A + np.eye(2)).round(6))

total = 26 / 14
for w1, w2 in ((total, 0.0), (total / 2, total / 2), (total + 5, -5.0)):
    print(f"w1 {w1:6.3f}  w2 {w2:6.3f}   predictions {X_dup @ np.array([w1, w2])}   w1² + w2² {w1 ** 2 + w2 ** 2:6.2f}")
print("Ridge λ = 1:", Ridge(alpha=1, fit_intercept=False).fit(X_dup, y3).coef_.round(3), "  26/29 =", round(26 / 29, 3))

det(XᵀX)      = 0.0   eigenvalues [ 0. 28.]
det(XᵀX + 1·I) = 29.0   eigenvalues [ 1. 29.]
w1  1.857  w2  0.000   predictions [1.8571 3.7143 5.5714]   w1² + w2²   3.45
w1  0.929  w2  0.929   predictions [1.8571 3.7143 5.5714]   w1² + w2²   1.72
w1  6.857  w2 -5.000   predictions [1.8571 3.7143 5.5714]   w1² + w2²  72.02
Ridge λ = 1: [0.897 0.897]   26/29 = 0.897


**What to look at.** The determinant is **0**, with eigenvalues 28 and 0: the zero is
the flat direction, raise $w_1$ and lower $w_2$ by the same amount and nothing changes.
Adding 1 to the diagonal moves both eigenvalues up by 1, to 29 and 1, and the
determinant becomes **29**. Three very different splits give the same predictions;
Ridge charges $w_1^2 + w_2^2$, smallest for the even split, and returns
$w_1 = w_2 = 26/29 = $ **0.897**.

**Try changing.** call `LinearRegression(fit_intercept=False).fit(X_dup, y3).coef_` and
see which of the many equivalent splits it reports.

## 15. Three coefficients, both penalties

In the simplest case, features uncorrelated and of unit length, each penalty acts
on each least-squares coefficient separately. Ridge **divides**, $w/(1+\lambda)$; Lasso
**subtracts and stops at zero**, $\operatorname{sign}(w)\max(|w| - \lambda, 0)$.

*From:* lesson 3, slide *Ridge shrinks, Lasso selects*; commentary, worked example *three coefficients, both penalties*.

In [29]:
X_id = np.eye(3)                     # three houses, three uncorrelated unit-length features
y_id = np.array([5.0, 2.0, 0.5])     # so least squares gives exactly 5, 2, 0.5

### Your prediction, before you run the next cell

At $\lambda = 3$, how many of the three coefficients will Ridge keep, and how
many will Lasso keep?

In [30]:
for lam in (0, 1, 3):
    ridge = Ridge(alpha=lam, fit_intercept=False).fit(X_id, y_id).coef_ if lam else y_id
    # scikit-learn's Lasso divides the squared error by 2m (here m = 3), so its alpha is λ/3
    lasso = Lasso(alpha=lam / 3, fit_intercept=False).fit(X_id, y_id).coef_ if lam else y_id
    print(f"λ = {lam}:  Ridge {ridge.round(3)}   Lasso {lasso.round(3)}")

λ = 0:  Ridge [5.  2.  0.5]   Lasso [5.  2.  0.5]
λ = 1:  Ridge [2.5  1.   0.25]   Lasso [4. 1. 0.]
λ = 3:  Ridge [1.25  0.5   0.125]   Lasso [2. 0. 0.]


**What to look at.** Ridge keeps all three, smaller (1.25, 0.5, 0.125 at
$\lambda = 3$). Lasso drops the smallest first and by $\lambda = 3$ keeps one: **2, 0, 0**.
The order it drops them in is the order of how much each was worth. One practical
trap: scikit-learn's `Lasso` writes its cost with a $\frac{1}{2m}$ in front of the squared
error, `Ridge` without, so the same `alpha` means different penalties in the two
classes.

**Try changing.** set $\lambda = 0.5$ and see whether Lasso already removes a
coefficient.

## 16. The coefficients are noise

The housing data of lesson 3 with the area entered twice, in square metres and in
square feet (with a rounding error, as a real conversion would have). The code loads it
from lesson 3's own data module and refits on four random splits, as the slide does.

*From:* lesson 3, slide *The coefficients are noise* and notebook 3, section *The case Ridge was invented for*.

In [31]:
sys.path.insert(0, "../Lessons/03_regression/Notebooks")
from housing_data import load_with_collinearity

houses = load_with_collinearity()
features = [c for c in houses.columns if c != "price"]
print("correlation of the two area columns:", round(houses["area_sqm"].corr(houses["area_sqft"]), 6))

correlation of the two area columns: 0.999999


### Your prediction, before you run the next cell

Across four splits, will the area coefficient stay within 10% of itself?
And what a square metre adds in total, `area_sqm` plus 10.76 × `area_sqft`?

In [32]:
print("split   area_sqm  area_sqft   one square metre in total")
for seed in range(4):
    Xa, _, ya, _ = train_test_split(houses[features], houses["price"], test_size=0.25, random_state=seed)
    w = LinearRegression().fit(Xa, ya).coef_
    print(f"{seed:5} {w[0]:10,.0f} {w[1]:10,.0f}   {w[0] + 10.7639 * w[1]:10,.0f}")

split   area_sqm  area_sqft   one square metre in total
    0      9,261       -640        2,371
    1     14,535     -1,127        2,405
    2     45,307     -3,989        2,373
    3     39,061     -3,402        2,442


**What to look at.** The square-metre coefficient runs from about 9,000 to 45,000,
and its duplicate swings the other way to compensate: the slide's table. Their combined
effect, what one square metre adds, stays within 2% of the true 2,400 on every
split. The **predictions** are fine; the **explanation** is noise. Anyone reading
these coefficients to learn what drives prices would draw nonsense, confidently.

### Your prediction, before you run the next cell

Ridge with $\lambda = 10$ on scaled features, same four splits. Stable or
not?

In [33]:
for seed in range(4):
    Xa, _, ya, _ = train_test_split(houses[features], houses["price"], test_size=0.25, random_state=seed)
    w = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xa, ya)[-1].coef_
    print(f"{seed:5} {w[0]:10,.0f} {w[1]:10,.0f}")

    0     39,279     39,270
    1     40,573     40,554
    2     37,958     37,883
    3     40,697     40,634


**What to look at.** Stable and nearly equal on both columns: two moderate
coefficients cost less under a squared penalty than one huge positive and one huge
negative. These are euros per standard deviation of area, because Ridge ran on scaled
features, so they cannot be compared with the table above; what matters is that they
stopped moving.

**Try changing.** drop `area_sqft` and refit: one column, one stable coefficient.

## 17. The same column, two coefficients

A coefficient is the change in the prediction for one more unit of its feature,
**with every other feature held fixed**. Which features are in the model changes what
"held fixed" means, and so changes the coefficient.

*From:* lesson 3, slide *What a coefficient actually says*; commentary, worked example *the same column, two coefficients*.

In [34]:
x1 = np.array([1.0, 2.0, 3.0, 4.0])
x2 = np.array([1.0, 3.0, 3.0, 5.0])
price_exact = 2 * x1 + 3 * x2              # the truth: 2 per unit of x1, 3 per unit of x2, no noise
print("correlation of x1 and x2:", round(np.corrcoef(x1, x2)[0, 1], 2))

correlation of x1 and x2: 0.95


### Your prediction, before you run the next cell

Fit with both features, then with $x_1$ alone. What coefficient will $x_1$
get each time?

In [35]:
both = LinearRegression().fit(np.column_stack([x1, x2]), price_exact).coef_
alone = LinearRegression().fit(x1.reshape(-1, 1), price_exact).coef_
print("both features:", both.round(3))
print("x1 alone:     ", alone.round(3))

both features: [2. 3.]
x1 alone:      [5.6]


**What to look at.** With both, **2** and **3**: the truth. With $x_1$ alone, **5.6**.
Neither fit is wrong. With $x_2$ in the model, 2 answers "one more unit of $x_1$, $x_2$
held fixed"; without it, 5.6 answers "one more unit of $x_1$, and whatever $x_2$
usually does alongside it". Lesson 3's 2,785 euros per square metre (area alone)
against 2,421 (all six features) is this, on the real data.

**Try changing.** make `x2` independent of `x1`, for example `[3, 1, 5, 3]`, and see
whether the coefficient of $x_1$ alone still moves.

## 18. When can you trust a coefficient?

A coefficient estimated from noisy data wobbles from one sample to the next. The
wobble grows when its feature is correlated with another: the two can trade the effect
between them. Here 500 datasets of 50 houses each are drawn from the same truth, and
the spread of the estimated coefficient is measured.

*From:* lesson 3, slide *When can you trust a coefficient?* and notebook 1, section *All six features*.

### Your prediction, before you run the next cell

The coefficient of the first feature is 2 in truth. As the correlation
with the second feature rises from 0 to 0.95, how much wider will the spread of its
estimate become: about the same, twice, or more than three times?

In [36]:
rng = np.random.default_rng(1)
for r in (0.0, 0.5, 0.9, 0.95):
    estimates = []
    for _ in range(500):
        a = rng.normal(size=50)
        c = r * a + np.sqrt(1 - r ** 2) * rng.normal(size=50)      # correlated r with a
        target = 2 * a + 3 * c + rng.normal(scale=1.0, size=50)
        estimates.append(LinearRegression().fit(np.column_stack([a, c]), target).coef_[0])
    estimates = np.array(estimates)
    print(f"r = {r:4}:  mean estimate {estimates.mean():.2f}   spread (standard deviation) {estimates.std():.3f}"
          f"   1/sqrt(1 - r²) = {1 / np.sqrt(1 - r ** 2):.2f}")

r =  0.0:  mean estimate 2.01   spread (standard deviation) 0.147   1/sqrt(1 - r²) = 1.00


r =  0.5:  mean estimate 2.00   spread (standard deviation) 0.164   1/sqrt(1 - r²) = 1.15
r =  0.9:  mean estimate 2.00   spread (standard deviation) 0.355   1/sqrt(1 - r²) = 2.29


r = 0.95:  mean estimate 2.01   spread (standard deviation) 0.493   1/sqrt(1 - r²) = 3.20


**What to look at.** The estimate stays centred on 2 at every correlation: it is not
biased. But its spread grows roughly as $1/\sqrt{1 - r^2}$, the last column, about three
times wider at 0.95 than at 0. On lesson 3's houses, bedrooms (correlated 0.77 with
area) came out 17% from the truth while distance and age, uncorrelated with area, were
within 1%. On real data the truth is unknown and the wobble is invisible: you see one
number.

**Try changing.** raise the houses per dataset from 50 to 500 and see how much of the
spread a larger sample removes.